# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # The metadata is an object, not a dict

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Date Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets and fields. All references will be by `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name} (ID: {rs.id})")
    if hasattr(rs, 'fields'):
        print("  Field Names and IDs:")
        for field in rs.fields:
            # Each field typically has .id and .name
            print(f"    - {getattr(field, 'name', '(unnamed)')} (@id: {field.id})")
    print()

# For reference, collect the first record set id for use later
record_set_ids = [r.id for r in record_sets]
print('Record set @id list:', record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
dfs = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dfs[record_set_id] = df
    print(f"Shape: {df.shape}; Columns: {df.columns.tolist()}")
    if not df.empty:
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping by key attributes.

**Note:** In the absence of specifics about the fields, we'll search for a typical numeric field by type or pattern (e.g., molecular weight, peptide count, etc.), and demonstrate EDA.

In [ ]:
# Let's pick the first (primary) record set for EDA
target_record_set_id = record_set_ids[0]
df = dfs[target_record_set_id]

# Display column names for reference
print("Record set columns:")
print(df.columns.tolist())

# Try to auto-select a numeric field (e.g., by known names)
possible_numeric_fields = [
    c for c in df.columns if any(x in c.lower() for x in ['mw', 'count', 'abundance', 'number', 'coverage', 'pI', 'intensity'])
]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # fallback: first numeric dtype field
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
    else:
        raise ValueError("No numeric field found!")

print(f"Chosen numeric field for EDA: {numeric_field}")
# Make sure the field is numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
df = df.dropna(subset=[numeric_field])

threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a group field if available (e.g., accession, sample, or description)
possible_group_fields = [
    c for c in df.columns if any(x in c.lower() for x in ['accession', 'sample', 'group', 'description', 'id']) and c != numeric_field
]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No group field selected for grouping step.")

## 5. Visualization
Visualize distributions or relationships in the filtered data using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a group field was found, visualize the group means
if 'group_field' in locals():
    plt.figure(figsize=(10, 5))
    top_grouped = grouped_df.sort_values(numeric_field, ascending=False).head(20)
    sns.barplot(
        x=top_grouped[numeric_field],
        y=top_grouped[group_field],
        orient='h'
    )
    plt.title(f"Top 20 {group_field}s by mean {numeric_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook has loaded and explored the FAIR<sup>2</sup> mass spectrometry dataset by Croissant schema, using the `mlcroissant` library. We listed record sets by `@id`, loaded their contents, performed basic exploratory analysis, and visualized distributions of key numeric fields, all using semantic identifiers. For deeper analysis, consult the Croissant schema or metadata documentation to explore all available fields and their scientific interpretation.